# Test API Credentials

This notebook validates `HF_TOKEN`, `ANTHROPIC_API_KEY`, `ALIBABA_API_KEY`, `GOOGLE_AI_API_KEY`, and `OPENAI_API_KEY` without displaying their values. It works with persistent PowerShell user variables on Windows and environment variables injected into a RunPod Pod.

- Hugging Face is tested with the authenticated `whoami` endpoint. This directly verifies the token and reports the account name.
- The four model providers are tested with a minimal text request asking for `OK`.
- Each test is isolated, so one failure does not stop the others.
- The default model IDs can be overridden with the variables in the configuration cell.

Running the live-test cell may incur very small API charges.

## 1. Install dependencies

In [ ]:
%pip install -q anthropic google-genai huggingface_hub openai pandas


## 2. Load credentials and configure models

On Windows, the helper checks both the current process environment and persistent user-level environment variables saved through PowerShell.

RunPod secrets must be mapped into the Pod template's **Environment Variables** section. Secrets stored in the Secrets page are not automatically placed in the container environment. Use these mappings:

| Environment variable | RunPod template value |
|---|---|
| `HF_TOKEN` | `{{ RUNPOD_SECRET_hf_token }}` |
| `ANTHROPIC_API_KEY` | `{{ RUNPOD_SECRET_anthropic_api_key }}` |
| `ALIBABA_API_KEY` | `{{ RUNPOD_SECRET_alibaba_api_key }}` |
| `GOOGLE_AI_API_KEY` | `{{ RUNPOD_SECRET_google_ai_api_key }}` |
| `OPENAI_API_KEY` | `{{ RUNPOD_SECRET_openai_api_key }}` |

The loader also accepts the lowercase names directly if you choose those as the environment-variable keys.

In [ ]:
import os
import platform

import pandas as pd

CREDENTIAL_ALIASES = {
    "HF_TOKEN": ("HF_TOKEN", "hf_token"),
    "ANTHROPIC_API_KEY": ("ANTHROPIC_API_KEY", "anthropic_api_key"),
    "ALIBABA_API_KEY": ("ALIBABA_API_KEY", "alibaba_api_key"),
    "GOOGLE_AI_API_KEY": ("GOOGLE_AI_API_KEY", "google_ai_api_key"),
    "OPENAI_API_KEY": ("OPENAI_API_KEY", "openai_api_key"),
}


def get_saved_environment_variable(names: tuple[str, ...]) -> tuple[str | None, str | None]:
    for name in names:
        value = os.getenv(name)
        if value:
            return value.strip(), f"process:{name}"

    if platform.system() == "Windows":
        import winreg

        for name in names:
            try:
                with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as key:
                    value, _ = winreg.QueryValueEx(key, name)
                    if value:
                        return str(value).strip(), f"windows-user:{name}"
            except (FileNotFoundError, OSError):
                continue
    return None, None


resolved_credentials = {
    canonical_name: get_saved_environment_variable(aliases)
    for canonical_name, aliases in CREDENTIAL_ALIASES.items()
}
credentials = {name: resolved[0] for name, resolved in resolved_credentials.items()}
credential_sources = {name: resolved[1] for name, resolved in resolved_credentials.items()}

# Override these defaults here or through environment variables when needed.
OPENAI_TEST_MODEL = os.getenv("OPENAI_TEST_MODEL", "gpt-4.1-nano")
ANTHROPIC_TEST_MODEL = os.getenv("ANTHROPIC_TEST_MODEL", "claude-haiku-4-5-20251001")
GOOGLE_TEST_MODEL = os.getenv("GOOGLE_TEST_MODEL", "gemini-2.5-flash-lite")
ALIBABA_TEST_MODEL = os.getenv("ALIBABA_TEST_MODEL", "qwen-flash")
ALIBABA_BASE_URL = os.getenv(
    "ALIBABA_BASE_URL",
    "https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

credential_status = pd.DataFrame(
    {
        "credential": list(CREDENTIAL_ALIASES),
        "accepted_names": [", ".join(names) for names in CREDENTIAL_ALIASES.values()],
        "found": [bool(credentials[name]) for name in CREDENTIAL_ALIASES],
        "found_as": [credential_sources[name] or "not found" for name in CREDENTIAL_ALIASES],
    }
)
credential_status


## 3. Run live authentication tests

In [ ]:
from collections.abc import Callable
from typing import Any

TEST_PROMPT = "Reply with exactly OK and no other text."


def sanitize(value: Any) -> str:
    text = str(value or "")
    for secret in credentials.values():
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text[:1000]


def test_hugging_face() -> str:
    from huggingface_hub import HfApi

    identity = HfApi().whoami(token=credentials["HF_TOKEN"])
    account = identity.get("name") or identity.get("fullname") or "authenticated account"
    return f"Authenticated as {account}"


def test_openai() -> str:
    from openai import OpenAI

    client = OpenAI(api_key=credentials["OPENAI_API_KEY"])
    response = client.responses.create(
        model=OPENAI_TEST_MODEL,
        input=TEST_PROMPT,
        max_output_tokens=16,
    )
    return response.output_text.strip()


def test_anthropic() -> str:
    from anthropic import Anthropic

    client = Anthropic(api_key=credentials["ANTHROPIC_API_KEY"])
    response = client.messages.create(
        model=ANTHROPIC_TEST_MODEL,
        max_tokens=8,
        messages=[{"role": "user", "content": TEST_PROMPT}],
    )
    return "".join(
        block.text for block in response.content if getattr(block, "type", None) == "text"
    ).strip()


def test_google() -> str:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=credentials["GOOGLE_AI_API_KEY"])
    response = client.models.generate_content(
        model=GOOGLE_TEST_MODEL,
        contents=TEST_PROMPT,
        config=types.GenerateContentConfig(max_output_tokens=8, temperature=0),
    )
    return (response.text or "").strip()


def test_alibaba() -> str:
    from openai import OpenAI

    client = OpenAI(
        api_key=credentials["ALIBABA_API_KEY"],
        base_url=ALIBABA_BASE_URL,
    )
    response = client.chat.completions.create(
        model=ALIBABA_TEST_MODEL,
        messages=[{"role": "user", "content": TEST_PROMPT}],
        max_tokens=8,
        temperature=0,
    )
    return (response.choices[0].message.content or "").strip()


tests: list[tuple[str, str, str, Callable[[], str]]] = [
    ("Hugging Face", "HF_TOKEN", "whoami", test_hugging_face),
    ("OpenAI", "OPENAI_API_KEY", OPENAI_TEST_MODEL, test_openai),
    ("Anthropic", "ANTHROPIC_API_KEY", ANTHROPIC_TEST_MODEL, test_anthropic),
    ("Google AI", "GOOGLE_AI_API_KEY", GOOGLE_TEST_MODEL, test_google),
    ("Alibaba Model Studio", "ALIBABA_API_KEY", ALIBABA_TEST_MODEL, test_alibaba),
]

results = []
for provider, variable, test_target, test_function in tests:
    if not credentials[variable]:
        results.append(
            {
                "provider": provider,
                "environment_variable": variable,
                "test_target": test_target,
                "status": "missing",
                "response_or_error": "Environment variable was not found.",
            }
        )
        continue

    try:
        response_text = test_function()
        results.append(
            {
                "provider": provider,
                "environment_variable": variable,
                "test_target": test_target,
                "status": "ok",
                "response_or_error": sanitize(response_text),
            }
        )
    except Exception as exc:
        results.append(
            {
                "provider": provider,
                "environment_variable": variable,
                "test_target": test_target,
                "status": "error",
                "response_or_error": sanitize(f"{type(exc).__name__}: {exc}"),
            }
        )

results_df = pd.DataFrame(results)
results_df


## Interpreting results

- `ok`: the credential authenticated and the requested endpoint responded.
- `missing`: the notebook could not find that environment variable.
- `error`: inspect the sanitized message. A valid key can still receive an authorization, billing, regional-endpoint, quota, or model-access error.

For Alibaba, change `ALIBABA_BASE_URL` if the key belongs to a different Model Studio region. No credential values are printed.

If all RunPod credentials show `missing`, verify that the five Secrets were referenced from the Pod template's Environment Variables section, then restart the Pod so the variables are injected into the container.